## **Tutorial 3 Part A Data Recording**
In this module, you have only limited access to the robots, making the implementation of algorithms highly challenging. One way to mitigate this challenge is by recording data. The ZED 2 SDK provides a convenient way to record RGB-D data in SVO2 format. This tutorial will guide you through the process of recording and reading RGB-D data.

To read an SVO2 file, you will still need to install the ZED 2 driver and its Python library. However, if you only need the color and depth images, you can remove this limitation by converting the file to .avi (RGB video) and .bin (depth data) formats. I have also included code in this tutorial for the conversion.

**Step 1 Creating Widgets for Display**

In [21]:
import ipywidgets.widgets as widgets
from IPython.display import display

#create two widgets for the displaying of the image
display_color = widgets.Image(format='jpeg', width='45%') #determine the width of the color image
display_depth = widgets.Image(format='jpeg', width='45%')  #determine the width of the depth image
layout=widgets.Layout(width='100%')

sidebyside = widgets.HBox([display_color, display_depth],layout=layout) #horizontal display

#Convert a NumPy array to JPEG-encoded data for display
def bgr8_to_jpeg(value):
    return bytes(cv2.imencode('.jpg',value)[1])

**Step 2 Record data into svo2 formate**

Note that Jetson Orin Nano doesn't have hardware encoder, you will have to use LOSSLESS format. You can reduce the file size to only 1% if using H265 encoding. 

In [22]:
# display the widget
display(sidebyside) 

#code modified from https://www.stereolabs.com/docs/video/recording
import pyzed.sl as sl
import cv2

# Create a ZED camera object
zed = sl.Camera()

# Create a InitParameters object and set configuration parameters
init = sl.InitParameters()
init.camera_resolution = sl.RESOLUTION.VGA #VGA(672*376), HD720(1280*720), HD1080 (1920*1080) or ...
init.depth_mode = sl.DEPTH_MODE.ULTRA  # Use ULTRA depth mode
init.coordinate_units = sl.UNIT.MILLIMETER  # Use meter units (for depth measurements)

#open zed camera
status = zed.open(init) 
if status != sl.ERROR_CODE.SUCCESS: 
    print("Zed Open", status, "Exit program.")
    exit(1)

#save the data in svo2 format (highly recommended)
recording_param = sl.RecordingParameters('test.svo2', sl.SVO_COMPRESSION_MODE.LOSSLESS) #jetson orin nano only support LOSSLESS ...
err = zed.enable_recording(recording_param)
if err != sl.ERROR_CODE.SUCCESS:
    print("Recording ZED : ", err)
    exit(1)

# Create and set RuntimeParameters after opening the camera
runtime = sl.RuntimeParameters()
print("Recording") 

# Declare your sl.Mat matrices
image_zed = sl.Mat()
depth_zed = sl.Mat()

#Record 180 frames
Total_frame = 180
i = 0 #frame index
t1 = cv2.getTickCount() #keep a record of the time, fps = frames/total time
while i < Total_frame:
    if zed.grab(runtime) == sl.ERROR_CODE.SUCCESS : # Check that a new image is successfully acquired
        i += 1 
        # The following code is for display purposes. You can comment it out to achieve higher recording FPS.
        zed.retrieve_image(image_zed, sl.VIEW.LEFT)
        zed.retrieve_measure(depth_zed, sl.MEASURE.DEPTH)

        scale = 0.1 # reduce image size
        color_value = image_zed.get_data()
        color_rgb_value = color_value[:, :, :3] #RGBA -> RGB
        resized_color = cv2.resize(color_rgb_value, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
        display_color.value = bgr8_to_jpeg(resized_color)
        
        #Get the depth image and convert it into a color image (for display only)
        depth_image = depth_zed.get_data()
        depth_colormap = cv2.applyColorMap(cv2.convertScaleAbs(depth_image, alpha=0.03), cv2.COLORMAP_JET) 
        resized_depth = cv2.resize(depth_colormap, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
        display_depth.value = bgr8_to_jpeg(resized_depth)

zed.close()
total_time = (cv2.getTickCount() - t1) / cv2.getTickFrequency()
print('time, ', total_time, 'fps ', Total_frame/ total_time)

[2025-02-16 17:10:08 UTC][ZED][INFO] Logging level INFO
[2025-02-16 17:10:08 UTC][ZED][INFO] Logging level INFO
[2025-02-16 17:10:08 UTC][ZED][INFO] Logging level INFO
[2025-02-16 17:10:09 UTC][ZED][INFO] [Init]  Depth mode: ULTRA
[2025-02-16 17:10:10 UTC][ZED][INFO] [Init]  Camera successfully opened.
[2025-02-16 17:10:10 UTC][ZED][INFO] [Init]  Camera FW version: 1523
[2025-02-16 17:10:10 UTC][ZED][INFO] [Init]  Video mode: VGA@100
[2025-02-16 17:10:10 UTC][ZED][INFO] [Init]  Serial Number: S/N 30918575
[2025-02-16 17:10:10 UTC][ZED][INFO] [Init]  Notice: The recording is using SVO version 2, enabled by default starting from SDK version 4.1. To revert to the original SVO version, set the environment variable "ZED_SDK_SVO_VERSION" to 1
Recording
time,  6.647891496 fps  27.076254194026035


In [24]:
# display the widget
display(sidebyside) 

import pyzed.sl as sl
import cv2
import math

# Create a ZED camera object
zed = sl.Camera()

# Set SVO path for playback
input_path = 'test.svo2'
init_params = sl.InitParameters()
init_params.depth_mode = sl.DEPTH_MODE.ULTRA
init_params.set_from_svo_file(input_path)

# Open the svo file
status = zed.open(init_params)
if status != sl.ERROR_CODE.SUCCESS: #Ensure the camera has opened succesfully
    print("Camera Open : "+repr(status)+". Exit program.")
    zed.close()
    exit(1)

# Set runtime parameters after opening the camera
runtime = sl.RuntimeParameters()

# Declare your sl.Mat matrices
image_zed = sl.Mat()
depth_zed = sl.Mat()
point_cloud = sl.Mat()

t1 = cv2.getTickCount()

while 1 :
  err = zed.grab(runtime)
  if err == sl.ERROR_CODE.SUCCESS :
    # Retrieve the left image, depth image, and point clouds
    zed.retrieve_image(image_zed, sl.VIEW.LEFT)
    zed.retrieve_measure(depth_zed, sl.MEASURE.DEPTH)
    zed.retrieve_measure(point_cloud, sl.MEASURE.XYZRGBA)

    scale = 0.5 # reduce image size
    color_value = image_zed.get_data()
    color_rgb_value = color_value[:, :, :3] #RGBA -> RGB
    resized_color = cv2.resize(color_rgb_value, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
      
    #draw a circle at the center point of the color image
    height,width,_ = color_value.shape
    cv2.circle(resized_color, (int(width*scale//2),int(height*scale//2)), 2, (0, 255, 0))
    display_color.value = bgr8_to_jpeg(resized_color)
    
    #Get the depth image and convert it into a color image (for display only)
    depth_image = depth_zed.get_data()
    depth_colormap = cv2.applyColorMap(cv2.convertScaleAbs(depth_image, alpha=0.03), cv2.COLORMAP_JET) 
    resized_depth = cv2.resize(depth_colormap, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    cv2.circle(resized_depth, (int(width*scale//2),int(height*scale//2)), 2, (0, 255, 0))
    display_depth.value = bgr8_to_jpeg(resized_depth)

    #get the 3D position of the center point of the image
    x = round(width / 2)
    y = round(height / 2)
    err, point_cloud_value = point_cloud.get_value(x, y) 

    # if math.isfinite(point_cloud_value[2]):
    #   print(point_cloud_value[0],point_cloud_value[1],point_cloud_value[2])
    #   distance = math.sqrt(point_cloud_value[0] * point_cloud_value[0] + point_cloud_value[1] * point_cloud_value[1] + point_cloud_value[2] * point_cloud_value[2])
    #   print('distance', distance)
    # else : 
    #   print("The distance can not be computed")
        
  elif zed.grab() == sl.ERROR_CODE.END_OF_SVOFILE_REACHED:
    print("SVO end has been reached")
    break

print( (cv2.getTickCount() - t1) / cv2.getTickFrequency())
zed.close()


[2025-02-16 17:10:47 UTC][ZED][INFO] Logging level INFO
[2025-02-16 17:10:47 UTC][ZED][INFO] Logging level INFO
[2025-02-16 17:10:47 UTC][ZED][INFO] Logging level INFO
[2025-02-16 17:10:47 UTC][ZED][INFO] [Init]  Depth mode: ULTRA
[2025-02-16 17:10:47 UTC][ZED][INFO] [Init]  Serial Number: S/N 30918575
SVO end has been reached
8.325623933
[2025-02-16 17:10:55 UTC][ZED][WARNING] END OF SVO FILE REACHED in sl::ERROR_CODE sl::Camera::grab(sl::RuntimeParameters)
[2025-02-16 17:10:55 UTC][ZED][WARNING] END OF SVO FILE REACHED in sl::ERROR_CODE sl::Camera::grab(sl::RuntimeParameters)


In [ ]:
# Convert the SVO2 file to AVI and BIN formats for better compatibility.
display(sidebyside) 

import pyzed.sl as sl
import cv2
import math
import numpy as np

# Create a ZED camera object
zed = sl.Camera()

# Set SVO path for playback
input_path = 'test.svo2'
init_params = sl.InitParameters()
init_params.depth_mode = sl.DEPTH_MODE.ULTRA
init_params.set_from_svo_file(input_path)

# Open the svo file
status = zed.open(init_params)
if status != sl.ERROR_CODE.SUCCESS: #Ensure the camera has opened succesfully
    print("Camera Open : "+repr(status)+". Exit program.")
    zed.close()
    exit(1)

# Set runtime parameters after opening the camera
runtime = sl.RuntimeParameters()

# Declare your sl.Mat matrices
image_zed = sl.Mat()
depth_zed = sl.Mat()
point_cloud = sl.Mat()

# Set the video codec
fourcc = cv2.VideoWriter_fourcc(*'XVID')  # 'mp4v' codec, suitable for MP4 files
width, height = 672,376 #VGA resolution
fps = 30
color_file = cv2.VideoWriter('color_video.avi', fourcc, fps, (width, height))
bin_file = open("depth_video.bin", "wb")

t1 = cv2.getTickCount()
while 1 :
  err = zed.grab(runtime)
  if err == sl.ERROR_CODE.SUCCESS :
    # Retrieve the left image, depth image, and point clouds
    zed.retrieve_image(image_zed, sl.VIEW.LEFT)
    zed.retrieve_measure(depth_zed, sl.MEASURE.DEPTH)

    # # use the get_data() method for a numpy array, only need the first three channels (RGB)
    color_value = image_zed.get_data()
    color_rgb_value = color_value[:, :, :3]
    color_file.write(color_rgb_value)

    #display color
    scale = 0.1 # reduce image size
    resized_color = cv2.resize(color_rgb_value, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    display_color.value = bgr8_to_jpeg(resized_color)

    #Get the depth image
    depth_np = depth_zed.get_data()
    depth_np = np.nan_to_num(depth_np, nan=0.0).astype(np.float32)  # Change NaN value to 0
    bin_file.write(depth_np.tobytes())

    #display depth
    depth_colormap = cv2.applyColorMap(cv2.convertScaleAbs(depth_np, alpha=0.03), cv2.COLORMAP_JET) 
    resized_depth = cv2.resize(depth_colormap, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    display_depth.value = bgr8_to_jpeg(resized_depth)
        
  elif zed.grab() == sl.ERROR_CODE.END_OF_SVOFILE_REACHED:
    print("SVO end has been reached")
    break

color_file.release()
bin_file.close()
zed.close()
print( (cv2.getTickCount() - t1) / cv2.getTickFrequency())



[2025-02-16 17:11:36 UTC][ZED][INFO] Logging level INFO
[2025-02-16 17:11:36 UTC][ZED][INFO] Logging level INFO
[2025-02-16 17:11:36 UTC][ZED][INFO] Logging level INFO
[2025-02-16 17:11:36 UTC][ZED][INFO] [Init]  Depth mode: ULTRA
[2025-02-16 17:11:36 UTC][ZED][INFO] [Init]  Serial Number: S/N 30918575
SVO end has been reached
[2025-02-16 17:11:47 UTC][ZED][WARNING] END OF SVO FILE REACHED in sl::ERROR_CODE sl::Camera::grab(sl::RuntimeParameters)
[2025-02-16 17:11:47 UTC][ZED][WARNING] END OF SVO FILE REACHED in sl::ERROR_CODE sl::Camera::grab(sl::RuntimeParameters)
12.888538615


In [ ]:
display(sidebyside) #display the widget

import cv2
import numpy as np
import time
width, height = 672,376
frame_size = width * height  # Number of pixels per frame
frame_bytes = frame_size * 4  # Bytes per frame (float32, 4 bytes per pixel)

#open the color video
cap = cv2.VideoCapture('color_video.avi')
if not cap.isOpened():
    print("Error: Could not open video file.")
    exit()

# Open the .bin file
f = open("depth_video.bin", "rb")
print("Starting depth video playback...")

t1=cv2.getTickCount()
while True:
    # Read frame data
    ret, color_img = cap.read()
    if not ret:
        print("End of video file.")
        break
    display_color.value = bgr8_to_jpeg(color_img)
    
    depth_data = f.read(frame_bytes)
    if not depth_data:
        break  # End of file, exit loop
    # Convert to NumPy array
    depth_image = np.frombuffer(depth_data, dtype=np.float32).reshape((height, width))
    depth_colormap = cv2.applyColorMap(cv2.convertScaleAbs(depth_image, alpha=0.03), cv2.COLORMAP_JET)
    display_depth.value = bgr8_to_jpeg(depth_colormap)
    time.sleep(0.001)

cap.release()
f.close()
print( (cv2.getTickCount() - t1) / cv2.getTickFrequency())


Starting depth video playback...
End of video file.
1.962603517
